# AIC 2026 — Build Milvus Vector Database

Notebook xây dựng vector database cho toàn bộ Batch 1 từ:

```text
manifest_keyframes.parquet
+
CLIP features .npy trong dataset Common
```

## Vì sao dùng Milvus Lite trên Kaggle?

Kaggle Notebook không phù hợp để chạy Docker Compose/Milvus Standalone.  
Milvus Lite chạy trực tiếp trong Python và lưu toàn bộ collection vào một file:

```text
aic2026_milvus.db
```

File này có thể được tải về và dùng trực tiếp trong web bằng `MilvusClient`.

## Input cần Add

1. Dataset/output chứa `manifest_keyframes.parquet` đã gộp 4 Part.
2. Dataset Common:

```text
/kaggle/input/datasets/minhdat27/aic2026-batch1-common
```

## Output

```text
/kaggle/working/aic2026_milvus_output/
├── aic2026_milvus.db
├── manifest_keyframes.parquet
├── milvus_config.json
├── build_report.json
└── README_WEB_INTEGRATION.md
```

`aic2026_milvus.db` là vector database dùng trong web.


In [1]:
# Cell 1 — Cài thư viện
!pip install -q -U "pymilvus[milvus-lite]" pyarrow tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.7/256.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 

In [2]:
# Cell 2 — Import và cấu hình
from __future__ import annotations

import gc
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pymilvus import MilvusClient, DataType

# ============================================================
# CẤU HÌNH
# ============================================================

# Dataset Common chứa clip-features-32.
COMMON_ROOT = Path("/kaggle/input/datasets/minhdat27/aic2026-batch1-common")

# Để None: notebook tự tìm manifest_keyframes.parquet trong /kaggle/input.
# Nếu có nhiều manifest, thay bằng path chính xác.
MANIFEST_PATH = Path(
    "/kaggle/input/notebooks/minhdat27/aic2026-merge-dataset/aic2026_batch1_merged/manifest_keyframes.parquet"
)
OUTPUT_DIR = Path("/kaggle/working/aic2026_milvus_output")
DB_PATH = OUTPUT_DIR / "aic2026_milvus.db"

COLLECTION_NAME = "aic2026_keyframes"
EMBEDDING_DIM = 512
METRIC_TYPE = "IP"

# Số entity gom lại trước mỗi lần insert.
INSERT_BATCH_SIZE = 5000

# True: xóa database cũ trong /kaggle/working và build lại từ đầu.
REBUILD_DATABASE = True

# True: chuẩn hóa L2 lại toàn bộ vector trước khi insert.
NORMALIZE_VECTORS = True

# Dùng để chạy thử nhanh; để None khi build chính thức.
MAX_ROWS: Optional[int] = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("COMMON_ROOT:", COMMON_ROOT)
print("DB_PATH:", DB_PATH)
print("COLLECTION_NAME:", COLLECTION_NAME)


COMMON_ROOT: /kaggle/input/datasets/minhdat27/aic2026-batch1-common
DB_PATH: /kaggle/working/aic2026_milvus_output/aic2026_milvus.db
COLLECTION_NAME: aic2026_keyframes


In [3]:
# Cell 3 — Kiểm tra input
from pathlib import Path

if not COMMON_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy Common dataset: {COMMON_ROOT}")

if MANIFEST_PATH is None or not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy manifest_keyframes.parquet:\n{MANIFEST_PATH}"
    )

CLIP_ROOT = (
    COMMON_ROOT
    / "clip-features-32-aic25-b1"
    / "clip-features-32"
)

if not CLIP_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục CLIP feature:\n{CLIP_ROOT}"
    )

print("COMMON_ROOT :", COMMON_ROOT)
print("MANIFEST    :", MANIFEST_PATH)
print("CLIP_ROOT   :", CLIP_ROOT)


COMMON_ROOT : /kaggle/input/datasets/minhdat27/aic2026-batch1-common
MANIFEST    : /kaggle/input/notebooks/minhdat27/aic2026-merge-dataset/aic2026_batch1_merged/manifest_keyframes.parquet
CLIP_ROOT   : /kaggle/input/datasets/minhdat27/aic2026-batch1-common/clip-features-32-aic25-b1/clip-features-32


In [4]:
# Cell 4 — Đọc và kiểm tra manifest
manifest = pd.read_parquet(MANIFEST_PATH)

if MAX_ROWS is not None:
    manifest = manifest.head(MAX_ROWS).copy()

required_columns = [
    "global_id",
    "video_id",
    "frame_id",
    "source_part",
    "feature_row",
    "clip_feature_relpath",
    "keyframe_relpath",
    "video_relpath",
    "timestamp_sec",
    "embedding_dim",
]

missing_columns = [
    column for column in required_columns
    if column not in manifest.columns
]

if missing_columns:
    raise RuntimeError(
        f"Manifest thiếu cột bắt buộc: {missing_columns}"
    )

manifest = (
    manifest
    .sort_values("global_id")
    .reset_index(drop=True)
)

checks = {
    "manifest_not_empty": len(manifest) > 0,
    "global_id_unique": bool(manifest["global_id"].is_unique),
    "vector_locator_unique": bool(
        ~manifest.duplicated(
            ["clip_feature_relpath", "feature_row"]
        ).any()
    ),
    "embedding_dim_512": bool(
        manifest["embedding_dim"].dropna().eq(EMBEDDING_DIM).all()
    ),
    "feature_rows_non_negative": bool(
        manifest["feature_row"].fillna(-1).ge(0).all()
    ),
    "frame_ids_present": bool(
        manifest["frame_id"].notna().all()
    ),
    "clip_relpaths_present": bool(
        manifest["clip_feature_relpath"].notna().all()
    ),
}

print(json.dumps(checks, ensure_ascii=False, indent=2))
print("Manifest rows:", f"{len(manifest):,}")
print("Videos:", f"{manifest['video_id'].nunique():,}")
print("Feature files:", f"{manifest['clip_feature_relpath'].nunique():,}")

critical_checks = [
    "manifest_not_empty",
    "global_id_unique",
    "vector_locator_unique",
    "embedding_dim_512",
    "feature_rows_non_negative",
    "clip_relpaths_present",
]

failed = [name for name in critical_checks if not checks[name]]

if failed:
    raise RuntimeError(
        "Manifest chưa đủ điều kiện build Milvus: "
        + ", ".join(failed)
    )

display(manifest.head())


{
  "manifest_not_empty": true,
  "global_id_unique": true,
  "vector_locator_unique": true,
  "embedding_dim_512": true,
  "feature_rows_non_negative": true,
  "frame_ids_present": true,
  "clip_relpaths_present": true
}
Manifest rows: 177,321
Videos: 873
Feature files: 873


,global_id,global_key,source_part,group_id,video_id,feature_row,keyframe_order,keyframe_id,keyframe_number,frame_id,...,media_info_path,media_info_relpath,title,author,publish_date,watch_url,image_ok,has_object,has_media_info,alignment_ok
0,0,part1:L25_V001:0,part1,L25,L25_V001,0,0,001,1,0,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L25_V001.json,BÍ QUYẾT ÔN THI THPT 2024 Môn Ngữ văn Chuyên ...,Báo Thanh Niên,15/06/2024,https://youtube.com/watch?v=XCraXMz1pY8,True,True,True,True
1,1,part1:L25_V001:1,part1,L25,L25_V001,1,1,002,2,68,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L25_V001.json,BÍ QUYẾT ÔN THI THPT 2024 Môn Ngữ văn Chuyên ...,Báo Thanh Niên,15/06/2024,https://youtube.com/watch?v=XCraXMz1pY8,True,True,True,True
2,2,part1:L25_V001:2,part1,L25,L25_V001,2,2,003,3,150,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L25_V001.json,BÍ QUYẾT ÔN THI THPT 2024 Môn Ngữ văn Chuyên ...,Báo Thanh Niên,15/06/2024,https://youtube.com/watch?v=XCraXMz1pY8,True,True,True,True
3,3,part1:L25_V001:3,part1,L25,L25_V001,3,3,004,4,300,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L25_V001.json,BÍ QUYẾT ÔN THI THPT 2024 Môn Ngữ văn Chuyên ...,Báo Thanh Niên,15/06/2024,https://youtube.com/watch?v=XCraXMz1pY8,True,True,True,True
4,4,part1:L25_V001:4,part1,L25,L25_V001,4,4,005,5,450,...,/kaggle/input/datasets/minhdat27/aic2026-batch...,media-info-aic25-b1/media-info/L25_V001.json,BÍ QUYẾT ÔN THI THPT 2024 Môn Ngữ văn Chuyên ...,Báo Thanh Niên,15/06/2024,https://youtube.com/watch?v=XCraXMz1pY8,True,True,True,True


In [5]:
# Cell 5 — Kiểm tra toàn bộ file .npy và mapping feature_row
validation_rows = []
missing_feature_files = []
invalid_feature_rows = []

for relpath, rows in tqdm(
    manifest.groupby("clip_feature_relpath", sort=False),
    desc="Validating CLIP files",
):
    feature_path = COMMON_ROOT / str(relpath)

    if not feature_path.exists():
        # Fallback nếu relpath chỉ tính từ CLIP_ROOT.
        fallback = CLIP_ROOT / Path(str(relpath)).name
        if fallback.exists():
            feature_path = fallback
        else:
            missing_feature_files.append(str(relpath))
            continue

    try:
        vectors = np.load(
            feature_path,
            mmap_mode="r",
            allow_pickle=False,
        )
    except Exception as exc:
        missing_feature_files.append(
            f"{relpath} | read error: {exc}"
        )
        continue

    if vectors.ndim != 2:
        raise RuntimeError(
            f"{feature_path} không phải ma trận 2 chiều: {vectors.shape}"
        )

    if vectors.shape[1] != EMBEDDING_DIM:
        raise RuntimeError(
            f"{feature_path} có dim={vectors.shape[1]}, "
            f"mong đợi {EMBEDDING_DIM}"
        )

    requested_rows = rows["feature_row"].astype(int).to_numpy()
    bad_rows = requested_rows[
        (requested_rows < 0)
        | (requested_rows >= vectors.shape[0])
    ]

    if len(bad_rows):
        invalid_feature_rows.append({
            "clip_feature_relpath": str(relpath),
            "vector_count": int(vectors.shape[0]),
            "invalid_rows": bad_rows[:20].tolist(),
        })

    validation_rows.append({
        "clip_feature_relpath": str(relpath),
        "feature_path": str(feature_path),
        "vector_count": int(vectors.shape[0]),
        "embedding_dim": int(vectors.shape[1]),
        "dtype": str(vectors.dtype),
        "manifest_rows": int(len(rows)),
        "max_requested_row": int(requested_rows.max())
            if len(requested_rows) else None,
    })

feature_validation = pd.DataFrame(validation_rows)

if missing_feature_files:
    print("Ví dụ file CLIP không tìm thấy:")
    for item in missing_feature_files[:20]:
        print(" -", item)
    raise FileNotFoundError(
        f"Thiếu {len(missing_feature_files)} file CLIP feature."
    )

if invalid_feature_rows:
    display(pd.DataFrame(invalid_feature_rows))
    raise RuntimeError(
        "Có feature_row vượt ngoài số vector trong file .npy."
    )

print("Tất cả CLIP feature đã hợp lệ.")
display(feature_validation.head())


Validating CLIP files:   0%|          | 0/873 [00:00<?, ?it/s]

Tất cả CLIP feature đã hợp lệ.


,clip_feature_relpath,feature_path,vector_count,embedding_dim,dtype,manifest_rows,max_requested_row
0,clip-features-32-aic25-b1/clip-features-32/L25...,/kaggle/input/datasets/minhdat27/aic2026-batch...,438,512,float16,438,437
1,clip-features-32-aic25-b1/clip-features-32/L25...,/kaggle/input/datasets/minhdat27/aic2026-batch...,352,512,float16,352,351
2,clip-features-32-aic25-b1/clip-features-32/L25...,/kaggle/input/datasets/minhdat27/aic2026-batch...,352,512,float16,352,351
3,clip-features-32-aic25-b1/clip-features-32/L25...,/kaggle/input/datasets/minhdat27/aic2026-batch...,345,512,float16,345,344
4,clip-features-32-aic25-b1/clip-features-32/L25...,/kaggle/input/datasets/minhdat27/aic2026-batch...,543,512,float16,543,542


In [6]:
# Cell 6 — Tạo Milvus Lite database và collection
if REBUILD_DATABASE and DB_PATH.exists():
    DB_PATH.unlink()
    print("Đã xóa database cũ:", DB_PATH)

client = MilvusClient(uri=str(DB_PATH))

if client.has_collection(collection_name=COLLECTION_NAME):
    if REBUILD_DATABASE:
        client.drop_collection(collection_name=COLLECTION_NAME)
    else:
        raise RuntimeError(
            f"Collection {COLLECTION_NAME} đã tồn tại. "
            "Đặt REBUILD_DATABASE=True để build lại."
        )

schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=False,
)

schema.add_field(
    field_name="id",
    datatype=DataType.INT64,
    is_primary=True,
)

schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=EMBEDDING_DIM,
)

schema.add_field(
    field_name="video_id",
    datatype=DataType.VARCHAR,
    max_length=64,
)

schema.add_field(
    field_name="frame_id",
    datatype=DataType.INT64,
)

schema.add_field(
    field_name="source_part",
    datatype=DataType.VARCHAR,
    max_length=16,
)

schema.add_field(
    field_name="feature_row",
    datatype=DataType.INT64,
)

schema.add_field(
    field_name="keyframe_relpath",
    datatype=DataType.VARCHAR,
    max_length=1024,
)

schema.add_field(
    field_name="video_relpath",
    datatype=DataType.VARCHAR,
    max_length=1024,
)

schema.add_field(
    field_name="timestamp_sec",
    datatype=DataType.FLOAT,
)

index_params = client.prepare_index_params()

# Milvus Lite dùng FLAT. Với vector L2-normalized, IP tương đương cosine.
index_params.add_index(
    field_name="embedding",
    index_type="FLAT",
    metric_type=METRIC_TYPE,
)

client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema,
    index_params=index_params,
)

print("Collection created:", COLLECTION_NAME)
print(client.describe_collection(COLLECTION_NAME))


Collection created: aic2026_keyframes
{'collection_name': 'aic2026_keyframes', 'auto_id': False, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 0, 'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 512}}, {'field_id': 0, 'name': 'video_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 64}}, {'field_id': 0, 'name': 'frame_id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}, {'field_id': 0, 'name': 'source_part', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 16}}, {'field_id': 0, 'name': 'feature_row', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}}, {'field_id': 0, 'name': 'keyframe_relpath', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 1024}}, {'field_id': 0, 'name': 'video_relpath', 'description': 

In [7]:
# Cell 7 — Insert vector và metadata theo batch
def safe_string(value, default="") -> str:
    if value is None or pd.isna(value):
        return default
    return str(value)


def safe_int(value, default=-1) -> int:
    if value is None or pd.isna(value):
        return default
    return int(value)


def safe_float(value, default=-1.0) -> float:
    if value is None or pd.isna(value):
        return default
    return float(value)


pending_records = []
inserted_count = 0
norm_stats = []

grouped = manifest.groupby(
    "clip_feature_relpath",
    sort=False,
)

for relpath, rows in tqdm(
    grouped,
    total=manifest["clip_feature_relpath"].nunique(),
    desc="Loading and inserting vectors",
):
    feature_path = COMMON_ROOT / str(relpath)

    if not feature_path.exists():
        feature_path = CLIP_ROOT / Path(str(relpath)).name

    vectors_all = np.load(
        feature_path,
        mmap_mode="r",
        allow_pickle=False,
    )

    row_indices = rows["feature_row"].astype(int).to_numpy()
    vectors = np.asarray(
        vectors_all[row_indices],
        dtype=np.float32,
    )

    norms_before = np.linalg.norm(vectors, axis=1)

    if NORMALIZE_VECTORS:
        safe_norms = np.clip(
            norms_before[:, None],
            1e-12,
            None,
        )
        vectors = vectors / safe_norms

    norms_after = np.linalg.norm(vectors, axis=1)

    norm_stats.append({
        "clip_feature_relpath": str(relpath),
        "count": int(len(vectors)),
        "mean_norm_before": float(norms_before.mean()),
        "mean_norm_after": float(norms_after.mean()),
    })

    rows_reset = rows.reset_index(drop=True)

    for i, row in rows_reset.iterrows():
        pending_records.append({
            "id": int(row["global_id"]),
            "embedding": vectors[i].tolist(),
            "video_id": safe_string(row["video_id"]),
            "frame_id": safe_int(row["frame_id"]),
            "source_part": safe_string(row["source_part"]),
            "feature_row": safe_int(row["feature_row"]),
            "keyframe_relpath": safe_string(
                row["keyframe_relpath"]
            ),
            "video_relpath": safe_string(
                row["video_relpath"]
            ),
            "timestamp_sec": safe_float(
                row["timestamp_sec"]
            ),
        })

        if len(pending_records) >= INSERT_BATCH_SIZE:
            result = client.insert(
                collection_name=COLLECTION_NAME,
                data=pending_records,
            )
            inserted_count += len(pending_records)
            pending_records.clear()

    del vectors
    del vectors_all
    gc.collect()

if pending_records:
    result = client.insert(
        collection_name=COLLECTION_NAME,
        data=pending_records,
    )
    inserted_count += len(pending_records)
    pending_records.clear()

try:
    client.flush(collection_name=COLLECTION_NAME)
except Exception:
    pass

print("Inserted:", f"{inserted_count:,}")
print("Expected:", f"{len(manifest):,}")

if inserted_count != len(manifest):
    raise RuntimeError(
        f"Số entity insert ({inserted_count}) "
        f"không bằng manifest ({len(manifest)})."
    )


Loading and inserting vectors:   0%|          | 0/873 [00:00<?, ?it/s]

Inserted: 177,321
Expected: 177,321


In [8]:
# Cell 8 — Load collection và kiểm tra số lượng entity
client.load_collection(
    collection_name=COLLECTION_NAME
)

entity_count = None

try:
    count_result = client.query(
        collection_name=COLLECTION_NAME,
        filter="",
        output_fields=["count(*)"],
    )
    print("Count query:", count_result)

    if count_result:
        entity_count = int(
            count_result[0].get("count(*)", -1)
        )
except Exception as exc:
    print("Count query không hỗ trợ ở phiên bản hiện tại:", exc)

if entity_count is not None and entity_count != len(manifest):
    raise RuntimeError(
        f"Milvus có {entity_count} entity, "
        f"manifest có {len(manifest)} dòng."
    )

print("Database file size:",
      f"{DB_PATH.stat().st_size / 1024**2:.2f} MB")


Count query: data: ["{'count(*)': 177321}"], extra_info: {}
Database file size: 0.00 MB


In [9]:
# Cell 9 — Smoke test: tìm chính vector đầu tiên
first_row = manifest.iloc[0]

first_feature_path = (
    COMMON_ROOT / str(first_row["clip_feature_relpath"])
)

if not first_feature_path.exists():
    first_feature_path = (
        CLIP_ROOT
        / Path(str(first_row["clip_feature_relpath"])).name
    )

first_vectors = np.load(
    first_feature_path,
    mmap_mode="r",
    allow_pickle=False,
)

query_vector = np.asarray(
    first_vectors[int(first_row["feature_row"])],
    dtype=np.float32,
)

if NORMALIZE_VECTORS:
    query_vector = query_vector / np.clip(
        np.linalg.norm(query_vector),
        1e-12,
        None,
    )

search_result = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector.tolist()],
    anns_field="embedding",
    limit=5,
    search_params={
        "metric_type": METRIC_TYPE,
        "params": {},
    },
    output_fields=[
        "video_id",
        "frame_id",
        "source_part",
        "feature_row",
        "keyframe_relpath",
        "video_relpath",
        "timestamp_sec",
    ],
)

print("Expected top-1 id:", int(first_row["global_id"]))
print("Search result:")
print(json.dumps(search_result, ensure_ascii=False, indent=2))

top1_id = int(search_result[0][0]["id"])

if top1_id != int(first_row["global_id"]):
    raise RuntimeError(
        "Smoke test thất bại: vector đầu tiên không trả về chính nó ở top-1."
    )

print("Smoke test PASSED.")


Expected top-1 id: 0
Search result:
[
  [
    {
      "id": 0,
      "distance": 1.0,
      "entity": {
        "id": 0,
        "video_id": "L25_V001",
        "frame_id": 0,
        "source_part": "part1",
        "feature_row": 0,
        "keyframe_relpath": "Keyframes_L25/keyframes/L25_V001/001.jpg",
        "video_relpath": "Videos_L25_a/video/L25_V001.mp4",
        "timestamp_sec": 0.0
      }
    },
    {
      "id": 438,
      "distance": 1.0,
      "entity": {
        "id": 438,
        "video_id": "L25_V002",
        "frame_id": 0,
        "source_part": "part1",
        "feature_row": 0,
        "keyframe_relpath": "Keyframes_L25/keyframes/L25_V002/001.jpg",
        "video_relpath": "Videos_L25_a/video/L25_V002.mp4",
        "timestamp_sec": 0.0
      }
    },
    {
      "id": 4242,
      "distance": 1.0,
      "entity": {
        "id": 4242,
        "video_id": "L25_V011",
        "frame_id": 0,
        "source_part": "part1",
        "feature_row": 0,
        "keyframe_re

In [10]:
# Cell 10 — Xuất database, manifest và báo cáo
# Copy manifest vào cùng output để web dùng làm metadata đầy đủ.
output_manifest = OUTPUT_DIR / "manifest_keyframes.parquet"
shutil.copy2(MANIFEST_PATH, output_manifest)

norms_df = pd.DataFrame(norm_stats)

build_report = {
    "schema_version": "aic2026-milvus-lite-v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "collection_name": COLLECTION_NAME,
    "database_file": DB_PATH.name,
    "manifest_file": output_manifest.name,
    "entity_count": int(len(manifest)),
    "video_count": int(manifest["video_id"].nunique()),
    "feature_file_count": int(
        manifest["clip_feature_relpath"].nunique()
    ),
    "embedding_dim": EMBEDDING_DIM,
    "metric_type": METRIC_TYPE,
    "index_type": "FLAT",
    "vectors_normalized": NORMALIZE_VECTORS,
    "mean_norm_before": float(
        np.average(
            norms_df["mean_norm_before"],
            weights=norms_df["count"],
        )
    ),
    "mean_norm_after": float(
        np.average(
            norms_df["mean_norm_after"],
            weights=norms_df["count"],
        )
    ),
    "database_size_bytes": int(DB_PATH.stat().st_size),
    "manifest_checks": checks,
    "smoke_test_passed": True,
}

milvus_config = {
    "milvus": {
        "mode": "lite",
        "uri": "data/milvus/aic2026_milvus.db",
        "collection_name": COLLECTION_NAME,
        "metric_type": METRIC_TYPE,
        "index_type": "FLAT",
        "embedding_dim": EMBEDDING_DIM,
    },
    "metadata": {
        "manifest_path":
            "data/processed/manifest_keyframes.parquet"
    },
}

with (
    OUTPUT_DIR / "build_report.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        build_report,
        f,
        ensure_ascii=False,
        indent=2,
    )

with (
    OUTPUT_DIR / "milvus_config.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        milvus_config,
        f,
        ensure_ascii=False,
        indent=2,
    )

readme = f"""# AIC 2026 Milvus Web Integration

## Generated assets

- `aic2026_milvus.db`: Milvus Lite database.
- `manifest_keyframes.parquet`: full metadata manifest.
- `milvus_config.json`: suggested project configuration.
- `build_report.json`: validation and build statistics.

## Copy into the project

```text
project/
├── data/
│   ├── milvus/
│   │   └── aic2026_milvus.db
│   └── processed/
│       └── manifest_keyframes.parquet
```

## Python connection

```python
from pymilvus import MilvusClient

client = MilvusClient(
    uri="data/milvus/aic2026_milvus.db"
)

results = client.search(
    collection_name="{COLLECTION_NAME}",
    data=[query_embedding],
    anns_field="embedding",
    limit=100,
    search_params={{
        "metric_type": "{METRIC_TYPE}",
        "params": {{}},
    }},
    output_fields=[
        "video_id",
        "frame_id",
        "source_part",
        "keyframe_relpath",
        "video_relpath",
        "timestamp_sec",
    ],
)
```

## Important

The web must encode text with the text encoder compatible with
the BTC-provided CLIP ViT-B/32 image features.

For a local single-machine competition system, Milvus Lite is sufficient.
For Docker Milvus Standalone, this `.db` file is not mounted directly;
the same records must be imported/inserted into the server collection.
"""

with (
    OUTPUT_DIR / "README_WEB_INTEGRATION.md"
).open("w", encoding="utf-8") as f:
    f.write(readme)

print(json.dumps(build_report, ensure_ascii=False, indent=2))
print("\nOutput files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(
        f" - {path.name}: "
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )


{
  "schema_version": "aic2026-milvus-lite-v1",
  "created_at_utc": "2026-08-02T17:53:20.728652+00:00",
  "collection_name": "aic2026_keyframes",
  "database_file": "aic2026_milvus.db",
  "manifest_file": "manifest_keyframes.parquet",
  "entity_count": 177321,
  "video_count": 873,
  "feature_file_count": 873,
  "embedding_dim": 512,
  "metric_type": "IP",
  "index_type": "FLAT",
  "vectors_normalized": true,
  "mean_norm_before": 1.00000036876855,
  "mean_norm_after": 1.0,
  "database_size_bytes": 4096,
  "manifest_checks": {
    "manifest_not_empty": true,
    "global_id_unique": true,
    "vector_locator_unique": true,
    "embedding_dim_512": true,
    "feature_rows_non_negative": true,
    "frame_ids_present": true,
    "clip_relpaths_present": true
  },
  "smoke_test_passed": true
}

Output files:
 - README_WEB_INTEGRATION.md: 0.00 MB
 - aic2026_milvus.db: 0.00 MB
 - build_report.json: 0.00 MB
 - manifest_keyframes.parquet: 2.40 MB
 - milvus_config.json: 0.00 MB


# Hoàn thành Data Layer

Sau khi notebook chạy thành công:

```text
Preprocess từng Part
→ Merge manifest
→ Build Milvus
→ Smoke test retrieval
```

thì phần **xử lý dữ liệu baseline đã hoàn thành**.

Bước tiếp theo không còn là preprocess nữa mà là:

```text
CLIP Text Encoder
→ Search Milvus
→ Hiển thị keyframe/video trên web
```

Sau đó mới nâng cấp chất lượng bằng:

- query translation/expansion;
- object reranking;
- OCR/ASR;
- Q&A;
- temporal alignment cho TRAKE;
- fine-tuning nếu có ground truth.
